In [36]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [37]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_1263/FORS2.2022-01-06T07_26_01.965/Mrk-1263_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_1263/FORS2.2022-01-06T07_26_01.965/Mrk-1263_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     359   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        40   (2094, 964)   float32   


In [38]:
image_cut = image[56:156, 0:1890]
image_error_cut = image_error[56:156, 0:1890]

In [39]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.00442
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [40]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) #+
        #ersic_1d(x_hr, params6) #+
        #sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [41]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_1263/FORS2.2022-01-06T07_26_01.965/PSF/Real_seeing_Mrk1263.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [42]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choco_area = single_integral(x, params5, a, b)
    #purple_area = single_integral(x, params6, a, b)
    #pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area + choco_area# + purple_area #+ pink_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area, (choco_area*100)/sum_area]#, (purple_area*100)/sum_area]#, (pink_area*100)/sum_area]

## OIII

In [43]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit_OIII = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_1263/FORS2.2022-01-06T07_26_01.965/Fit/OIII_fit.csv", index_col = 0)

In [44]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -sigma, fit_OIII['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -sigma, fit_OIII['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -sigma, fit_OIII['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -sigma, fit_OIII['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -sigma, fit_OIII['Component 5'].iloc[3] + sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -sigma, fit_OIII['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

38.99333728732655
30.70939405484414
6.1394646047161645
6.018115084933626
8.311341560581058
8.311113793919194
41.63142933624877
43.37341092885692
16.218759855468512
16.3929776894225
                  Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue        9.987143e+01  2.048241e+01  1.292843e+01   0.073013   0.094611
Green       2.028997e-32  6.601738e+01  2.027932e+00   0.000000   0.000000
Magenta    1.188423e-292  9.005755e+00  7.135751e+01   0.000000   0.000000
Orange      1.285682e-01  4.494458e+00  1.368613e+01  99.873764  61.924199
Chocolate   1.030220e-61  1.982634e-36  2.497509e-27   0.053223  37.981190


In [45]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -2*sigma, fit_OIII['Component 5'].iloc[3] + 2*sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -2*sigma, fit_OIII['Component 6'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

105.8717618751742
84.65679555663826
17.27982335005581
16.936348150448676
16.49154101414223
16.448948744478578
118.19880883274652
122.50747875656575
47.616951359658145
48.04999634075725
                  Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue        9.985671e+01  2.231874e+01  1.342454e+01   0.077587   0.097138
Green       6.340735e-19  6.128400e+01  6.772547e+00   0.000000   0.000000
Magenta    1.620883e-139  1.155409e+01  6.566957e+01   0.000000   0.000000
Orange      1.432915e-01  4.843165e+00  1.413334e+01  99.796527  65.138281
Chocolate   2.697795e-59  1.631521e-34  1.966438e-25   0.125887  34.764581


In [46]:
data1 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 5'].iloc[3] -3*sigma, fit_OIII['Component 5'].iloc[3] + 3*sigma)

#data6 = contamination(x_axes, fit_OIII['Component 1'], fit_OIII['Component 2'], fit_OIII['Component 3'], fit_OIII['Component 4'], fit_OIII['Component 5'], fit_OIII['Component 6'], fit_OIII['Component 6'].iloc[3] -3*sigma, fit_OIII['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

154.84237336773978
128.9135651273051
25.305005247733938
24.85281585878438
24.81992966398793
24.674044257992286
173.74010596579095
178.3092445874294
77.37185577181995
77.87247821680062
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       9.977710e+01  2.699045e+01  1.406579e+01   0.089324   0.100921
Green      8.135141e-06  4.976511e+01  1.611443e+01   0.000000   0.000000
Magenta    1.308702e-24  1.757310e+01  5.515560e+01   0.000000   0.000000
Orange     2.228941e-01  5.671354e+00  1.466417e+01  99.472908  72.412162
Chocolate  7.567741e-54  2.684822e-32  1.753258e-23   0.437768  27.486917


## HBeta

In [47]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_1263/FORS2.2022-01-06T07_26_01.965/Fit/Hbeta_fit.csv", index_col = 0)

In [48]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 1'].iloc[3] -sigma, fit_OIII['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 2'].iloc[3] -sigma, fit_OIII['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 3'].iloc[3] -sigma, fit_OIII['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 4'].iloc[3] -sigma, fit_OIII['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit_OIII['Component 5'].iloc[3] -sigma, fit_OIII['Component 5'].iloc[3] + sigma)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_OIII['Component 6'].iloc[3] -sigma, fit_OIII['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

6.051854025194172
6.010474002467922
1.8042855480116258
1.7638791733564987
6.035006145411322
4.849352043240952
9.656168059859386
9.003751113580872
5.3282819956196725
4.014487402385625
              Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       99.493024  28.572676   9.005257   0.351750   0.393972
Green       0.000000  52.346671   0.091330   0.000000   0.000000
Magenta     0.156883  13.754069  84.860969   0.040328   0.027332
Orange      0.343920   5.276938   5.997943  99.259050  53.371785
Chocolate   0.006173   0.049645   0.044500   0.348872  46.206911


In [49]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 1'].iloc[3] -2*sigma, fit_OIII['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 2'].iloc[3] -2*sigma, fit_OIII['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 3'].iloc[3] -2*sigma, fit_OIII['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 4'].iloc[3] -2*sigma, fit_OIII['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_OIII['Component 5'].iloc[3] -2*sigma, fit_OIII['Component 5'].iloc[3] + 2*sigma)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
#                     fit_OIII['Component 6'].iloc[3] -2*sigma, fit_OIII['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

16.935738661858984
16.892418151312413
5.261470600619374
5.050063880042186
9.720955801626173
8.158239786121847
27.364716524341738
25.6981806076151
15.2221015548841
11.851756920579342
              Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       99.450690  29.809828  11.383443   0.373492   0.414743
Green       0.000000  47.336578   0.775448   0.000000   0.000000
Magenta     0.171425  17.326290  80.193578   0.043254   0.028974
Orange      0.371252   5.476034   7.591745  99.190100  57.485160
Chocolate   0.006633   0.051270   0.055785   0.393154  42.071123


In [50]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 1'].iloc[3] -3*sigma, fit_OIII['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 2'].iloc[3] -3*sigma, fit_OIII['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 3'].iloc[3] -3*sigma, fit_OIII['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 4'].iloc[3] -3*sigma, fit_OIII['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                     fit_OIII['Component 5'].iloc[3] -3*sigma, fit_OIII['Component 5'].iloc[3] + 3*sigma)

#data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
#                     fit_OIII['Component 6'].iloc[3] -3*sigma, fit_OIII['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s-OIII.csv')
df.to_csv(df_path, float_format='%.3f')

27.36951324588858
27.546065533582542
8.394587455450857
7.848892904587725
12.254994102441863
10.722246432381734
40.468635936296714
38.44029773644972
23.799917382143803
19.65606151916528
                  Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue        9.921426e+01  32.371142  13.960699   0.424358   0.445074
Green      2.572118e-123  34.136960   3.854488   0.000000   0.000000
Magenta     2.633717e-01  27.586085  72.791529   0.050507   0.031683
Orange      5.136943e-01   5.851711   9.325836  98.970797  65.836942
Chocolate   8.670422e-03   0.054102   0.067448   0.554337  33.686302
